# V36 — Healthy-alternative spending / total food spending HMM

Set `INPUT_PATH`, then Restart Kernel and Run All. Dependencies: `numpy pandas scipy duckdb pyarrow tqdm`.

For trip $j$, household $i$, year $t$, and category $c$, the outcome is
$$Y_{cijt}=\frac{H_{cijt}}{F_{ijt}},\qquad F_{ijt}=\texttt{food\_spend}_{ijt}>0.$$

| Category | Healthy numerator | Denominator |
|---|---|---|
| FV | `fresh_spending` | `food_spend` |
| Milk | `skimmed_milk_spending` | `food_spend` |
| Yogurt | `skimmed_yogurt_spending` | `food_spend` |
| Cereal | `low_sugar_cereal_spending` | `food_spend` |
| Snack | `healthy_snack_spending` | `food_spend` |
| Beverage | `healthy_beverage_spending` | `food_spend` |

Healthy definitions are unchanged. The denominator is the existing input `food_spend` column, not a sum of the six categories. Old priority-coded outcomes and other-category spending columns are not used.

## What changes from V35
- Every positive-food-spending trip contributes all six shares. If no healthy alternative is bought in a category, its share is **zero**, including trips with no purchase in that category.
- Category-incidence regressions are removed. The state-dependent share distribution now models zeros as well as positive shares; a separate category-incidence factor would represent a different joint-outcome specification.
- Trips with `food_spend == 0` have undefined shares and contribute **no share emission**. They remain in the household-year panel and are counted in the audit, so the observed-year chain is preserved. A household-year with no eligible trips has a neutral emission and is inferred from its prior/transitions and other years.
- Negative/nonfinite spending or healthy spending exceeding `food_spend` by more than `1e-8` dollars causes a validation error. Positive excesses up to `1e-8` dollars are snapped to food spending in memory only; original Parquet values are unchanged. Corrections are exported in `floating_point_spending_audit.csv`.

## Share distribution and latent states
Retain V35's zero-and-one-inflated beta specification. Let $q=P(0<Y<1\mid S=s,X,F>0)$, $r=P(Y=1\mid Y\in\{0,1\},S=s,X,F>0)$, and $\mu=E[Y\mid0<Y<1,S=s,X,F>0]$.

$$f_c(y\mid s,X,F>0)=\begin{cases}
(1-q)(1-r),&y=0,\\
(1-q)r,&y=1,\\
q\,\operatorname{Beta}(y;\mu\phi_c,(1-\mu)\phi_c),&0<y<1.
\end{cases}$$

The interior-probability, boundary-one-probability, and beta-mean indices each have an intercept, a `state_num` coefficient ($s=1,2$), and the original four demographic controls. The beta mean uses the numerical link $10^{-9}+(1-2\cdot10^{-9})\operatorname{logit}^{-1}(\eta)$; only audited positive spending excesses up to `1e-8` dollars are corrected before constructing fractions. Precision is category-specific and shared across states; log-precision search bounds remain [-10,14] and boundary fits are flagged.

**Overall expected healthy-food share:** $(1-q)r+q\mu$.

A 100% share now means the trip's **entire food budget** was spent on that healthy alternative. This may never occur for some categories. If a category has no 100% observations, set $r=0$ and omit its boundary-one regression in **both** models. That category uses a zero-inflated beta distribution. This support rule is determined once from the retained sample and recorded in `share_data_audit.csv`; no rare observations are removed. Each category must have zero and interior observations to identify the retained model; otherwise validation stops for an explicit specification review.

The annual latent state, six event transition regressions, fixed common 2007 initial probability, 2008–2017 window, absence of an extra three-year filter, and observed-year gap policy are unchanged. No train/test split is introduced. The nested model has one class for each household's full observed history ($P=I_2$).

**Parameter counts:** if $m$ categories have 100% observations, nested has $78+6m$ parameters and proposed has $92+6m$. With no 100% observations these are **78 / 92**; with all six they are **114 / 128**. Fixed boundary probabilities and the fixed initial prior are not free parameters.

The emission multiplies six category densities conditional on state and covariates, retaining **conditional independence** from V35. The shared state induces marginal cross-category association; remaining within-state dependence is not modeled. The common denominator and budget constraints can create residual dependence. The product emission does not enforce a joint sum-to-one/budget constraint, so this remains a working model that should be checked for residual dependence. No category receives an arbitrary extra weight.

## Posterior healthy-share significance tests
For state $s$, let $\gamma_{its}=P(S_{it}=s\mid\text{all observed data})$ and $D_{ijt}=1\{F_{ijt}>0\}$. The main posterior mean is
$$\widehat M_{cs}=\frac{\sum_{it}\gamma_{its}\sum_j D_{ijt}Y_{cijt}}{\sum_{it}\gamma_{its}\sum_j D_{ijt}}.$$

The separately reported pooled food-budget share is
$$\widehat R_{cs}=\frac{\sum_{it}\gamma_{its}\sum_j H_{cijt}}{\sum_{it}\gamma_{its}\sum_j F_{ijt}}.$$
The first gives eligible trips equal weight; the second weights by food spending. They answer different questions.

For **each category and both models**, test $H_0:M_{c2}-M_{c1}=0$ with a two-sided Wald z-test, using household-clustered delta-method SEs. The difference SE includes covariance between the two state estimates because each household contributes to both. The same procedure is provided for $R$.

Exports include both state estimates and SEs, the signed difference, its SE and 95% CI, z-statistic, p-value, and stars: **\* p<.10, \*\* p<.05, \*\*\* p<.01**. Stars attach to the **State 2 minus State 1 difference**, not to tests of each positive share against zero. Percentage-point columns and a readable table are also exported. P-values are unadjusted across categories.

**Inference limitation:** posterior profile SEs treat fitted posterior weights as fixed. Coefficient SEs retain the block M-step Hessian approximation. Neither incorporates full state/parameter estimation uncertainty; use whole-household refitting bootstrap for final inference. Fitted model-implied expected shares use a common eligible-trip covariate distribution for both states and are exported as point estimates only. State labels must be checked against all six profiles.

LL/AIC/BIC compare proposed and nested **within V36** under its working emission specification. Do not compare these values directly with V35 or V30, which model different outcomes. BIC uses independent households; the no-switch restriction does not receive a standard chi-square likelihood-ratio p-value.


## Software validation performed before delivery
The full synthetic Parquet workflow passed for both models using **120 households, 480 household-years and 5,760 trips**, with zero-food trips, zero healthy-food shares, interior shares, and 100% FV shares. Both models met the EM convergence checks. The default fixture has one category with 100% observations: **98 proposed / 84 nested parameters**. A second synthetic run with no 100% shares in any category also converged: **92 proposed / 78 nested parameters**.

Independent checks verified:
- Annual sufficient-statistic emissions equal direct trip-by-trip healthy/food-spending beta-mixture densities, with no separate incidence likelihood.
- A household-year containing only zero-food trips remains in the panel and has a neutral emission.
- Forward likelihood equals explicit enumeration over all state paths.
- Both posterior mean trip shares and pooled spending shares match independent direct trip calculations.
- Household-clustered difference SEs include cross-state covariance; p-values, significance stars and confidence limits agree with independent calculations.
- Healthy spending more than `1e-8` dollars above food spending is rejected; smaller positive excesses are corrected consistently before share construction.
- Absent 100% components are handled consistently in both models, free-parameter counts are correct, fitted coefficient SEs are finite on the fixtures, accepted EM likelihoods are nondecreasing, and the common initial prior remains fixed.
- All requested exports and the notebook schema validate.

Synthetic results are software fixtures, not empirical evidence. The confidential full data were unavailable here. Run the notebook locally to obtain your actual estimates, significance tests and convergence diagnostics.


In [ ]:
import os

# Numerical-library thread settings MUST run before importing NumPy/SciPy.
# HMM_NUM_THREADS overrides the conservative default of min(4, detected CPUs).
_thread_text = os.environ.get("HMM_NUM_THREADS")
if _thread_text is None:
    HMM_NUM_THREADS = min(4, os.cpu_count() or 1)
else:
    try:
        HMM_NUM_THREADS = int(_thread_text)
    except (TypeError, ValueError) as exc:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.") from exc
    if HMM_NUM_THREADS < 1:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.")
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_variable] = str(HMM_NUM_THREADS)
print(f"Requested numerical threads = {HMM_NUM_THREADS}")
print("Thread settings take effect only before NumPy/SciPy import; after changing them, Restart Kernel and Run All.")

import gc
import math
import time
import warnings
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logsumexp
from scipy.stats import norm
try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        """Minimal progress fallback used only when tqdm is unavailable."""
        def __init__(self, iterable=None, total=None, desc="", **kwargs):
            self.iterable = iterable; self.total = total; self.desc = desc; self.n = 0
        def __enter__(self):
            print(self.desc); return self
        def __exit__(self, exc_type, exc, tb):
            return False
        def __iter__(self):
            for item in self.iterable:
                yield item; self.n += 1
        def update(self, value=1):
            self.n += value
        def set_description_str(self, text):
            self.desc = text; print(text)
        def set_postfix_str(self, text):
            pass
        def set_postfix(self, **kwargs):
            pass


import duckdb
import json
INPUT_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel.parquet")
OUTPUT_DIR = INPUT_PATH.parent / 'V36_Healthy_Food_Share_HMM_Results'
RUN_MODE = 'full_estimation'  # 'full_estimation', 'validate_only', or 'synthetic'
MAX_EM_ITERS = 150
SYNTHETIC_EM_ITERS = 40
MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 1  # No extra three-year household filter.
YEAR_GAP_POLICY = 'observed_step'  # 'error' rejects calendar-year gaps.
DUCKDB_MEMORY_LIMIT = '4GB'
CHECK_DUPLICATE_TRIPS = True
TOL=1e-3
EM_REL_TOL=1e-10
EM_STABLE_ITERS=3
EM_PRINT_EVERY=5
OPT_FTOL=1e-14
OPT_GTOL=1e-8
OPT_NEWTON_STEPS=20
SEED=20261030
FRESH_SHARE_2007_COL='initial_fresh_fv_share_2007'
TRANSITION_EVENT_COLUMNS=['event_new_child','getting_married','getting_divorce_or_separation','moving','getting_job','retirement']
TRANSITION_X_COLS=['intercept',*TRANSITION_EVENT_COLUMNS]
GATE_COVARIATE_COLS=['K_1','income10k_imputed','family_size_numeric','age']
GATE_X_COLS=['intercept','state_num',*GATE_COVARIATE_COLS]
SPENDING_ATOL = 1e-8  # Absolute dollar tolerance, matching the spending audit.
BETA_MEAN_EPS = 1e-9
LOG_PRECISION_BOUNDS = (-10.0, 14.0)
from scipy.special import betaln, digamma, polygamma
CATEGORY_CONFIG=OrderedDict([
 ('fv', dict(healthy='fresh_spending')),
 ('milk', dict(healthy='skimmed_milk_spending')),
 ('yogurt', dict(healthy='skimmed_yogurt_spending')),
 ('cereal', dict(healthy='low_sugar_cereal_spending')),
 ('snack', dict(healthy='healthy_snack_spending')),
 ('beverage', dict(healthy='healthy_beverage_spending'))])
CATEGORY_NAMES=list(CATEGORY_CONFIG)
np.random.seed(SEED)


## 1. Numerical engine
Retain V30's analytic logit gradients, warm starts and monotone EM safeguards. Add beta regression with analytic gradients and Hessians. Only annual sufficient statistics and unique covariate designs enter EM.


In [ ]:
@dataclass
class GateBlock:
    """Annual outcome counts mapped to one shared unique-covariate design."""

    key: str
    category: str
    prefix: str
    x_base: np.ndarray
    target: np.ndarray
    hy_pos: np.ndarray
    eligible_rows: int
    choice_weight: float
    success_count: np.ndarray
    failure_count: np.ndarray
    design_pos: np.ndarray


@dataclass
class ModelArrays:
    """NumPy model data with cached household-year and gate structures."""

    purchase: Dict[str, np.ndarray]
    outcome: Dict[str, np.ndarray]
    spend: Dict[str, np.ndarray]
    food_total_spend: np.ndarray
    food_total_spend_10: np.ndarray
    gate_covariates: Dict[str, np.ndarray]
    gate_blocks: Dict[str, GateBlock]
    trip_hy_pos: np.ndarray
    household_year: pd.DataFrame
    hy_key: pd.MultiIndex
    hh_blocks: Dict[str, Dict[str, np.ndarray]]
    households: List[str]
    years_by_household: Dict[str, List[int]]
    hh_starts: np.ndarray
    hh_ends: np.ndarray
    transition_x: np.ndarray
    transition_curr_pos: np.ndarray
    transition_row_of_hy: np.ndarray


@dataclass
class PreparedData:
    """Annual sufficient statistics and model buffers.

    df is a count-only RangeIndex shell carrying trip counts and the sample
    audit. Raw trips stay in DuckDB; EM uses exact annual sufficient statistics.
    The legacy incidence_models result field is empty in V36.
    """

    df: pd.DataFrame
    hy: pd.DataFrame
    arrays: ModelArrays
    annual_stats: pd.DataFrame


@dataclass
class EstimationResult:
    """Fitted model blocks and final optimizer diagnostics."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    beta_from1: np.ndarray
    beta_from2: np.ndarray
    pi_state2: float
    gamma_df: pd.DataFrame
    history_df: pd.DataFrame
    initial_state_table: pd.DataFrame
    transition_table: pd.DataFrame
    log_likelihood: float
    transition_fits: Optional[Dict[str, Dict[str, Any]]] = None
    optimizer_diagnostics_df: Optional[pd.DataFrame] = None


def safe_log(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Take a stable logarithm of probabilities."""
    return np.log(np.clip(x, eps, 1.0))


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Stable logistic transform."""
    return expit(x)


def significance_stars(p_value: float) -> str:
    """Return conventional significance stars."""
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


def ensure_columns(df: pd.DataFrame, required_cols: Sequence[str]) -> None:
    """Raise a clear error when required columns are missing."""
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def first_nonmissing(series: pd.Series) -> float:
    """Return the first observed value, preserving missingness when all values are missing."""
    observed = series.dropna()
    return observed.iloc[0] if len(observed) else np.nan


def _optimizer_diagnostics(
    result: Any,
    block_name: str,
    objective_value: float,
    gradient: np.ndarray,
    n_observations: int,
    effective_weight: float,
    covariance_success: bool,
    solver: str,
) -> Dict[str, Any]:
    return {
        "scaled_gradient_inf": float(getattr(result, 'scaled_gradient_inf', np.nan)),
        "initial_objective": float(getattr(result, 'initial_objective', np.nan)),
        "objective_improvement": float(getattr(result, 'objective_improvement', np.nan)),
        "reverted_to_start": bool(getattr(result, 'reverted_to_start', False)),
        "lbfgs_message": str(getattr(result, 'lbfgs_message', '')),
        "newton_refinements": int(getattr(result, 'newton_refinements', 0)),
        "model_block": block_name,
        "solver": solver,
        "optimizer_success": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "iterations": int(getattr(result, "nit", -1)),
        "function_evaluations": int(getattr(result, "nfev", -1)),
        "gradient_evaluations": int(getattr(result, "njev", -1)),
        "final_objective": float(objective_value),
        "gradient_norm": float(np.linalg.norm(gradient)),
        "n_observations": int(n_observations),
        "effective_weight": float(effective_weight),
        "covariance_hessian_success": bool(covariance_success),
    }


def _finish_logit_fit(
    result: Any,
    hessian_function: Any,
    gradient_function: Any,
    param_names: Sequence[str],
    block_name: str,
    objective_function: Any,
    n_observations: int,
    effective_weight: float,
    compute_se: bool,
    solver: str,
) -> Dict[str, Any]:
    beta = np.asarray(result.x, dtype=np.float64)
    covariance_success = False
    if compute_se:
        try:
            hessian = np.asarray(hessian_function(beta), dtype=np.float64)
            covariance = np.linalg.pinv(hessian)
            standard_error = np.sqrt(np.clip(np.diag(covariance), 0.0, np.inf))
            covariance_success = bool(np.isfinite(covariance).all())
        except Exception as exc:
            warnings.warn(f"{block_name}: Hessian/covariance failed: {exc}", RuntimeWarning)
            hessian = np.full((len(beta), len(beta)), np.nan)
            covariance = np.full((len(beta), len(beta)), np.nan)
            standard_error = np.full(len(beta), np.nan)
    else:
        hessian = np.full((len(beta), len(beta)), np.nan)
        covariance = np.full((len(beta), len(beta)), np.nan)
        standard_error = np.full(len(beta), np.nan)

    gradient = np.asarray(gradient_function(beta), dtype=np.float64)
    objective_value = float(objective_function(beta))
    with np.errstate(divide="ignore", invalid="ignore"):
        z_value = beta / standard_error
        p_value = 2.0 * (1.0 - norm.cdf(np.abs(z_value)))
    table = pd.DataFrame({
        "parameter": list(param_names),
        "estimate": beta,
        "std_error": standard_error,
        "z_value": z_value,
        "p_value": p_value,
    })
    table["significance_stars"] = table["p_value"].map(significance_stars)
    diagnostics = _optimizer_diagnostics(
        result, block_name, objective_value, gradient, n_observations,
        effective_weight, covariance_success, solver,
    )
    if not result.success:
        warnings.warn(
            f"Optimizer did not report success for {block_name}: "
            f"status={result.status}; message={result.message}",
            RuntimeWarning,
        )
    return {
        "_hessian_function": hessian_function,
        "result": result,
        "coef": beta,
        "hessian": hessian,
        "cov": covariance,
        "table": table,
        "nll": objective_value,
        "diagnostics": diagnostics,
    }


def _cached_value_gradient(callback):
    """Reuse one matrix product/sigmoid pass for both L-BFGS-B callbacks."""
    last_beta = None
    last_result = None
    def evaluate(beta):
        nonlocal last_beta, last_result
        if last_beta is None or not np.array_equal(beta, last_beta):
            last_beta = np.array(beta, copy=True)
            last_result = callback(beta)
        return last_result
    return evaluate


def _minimize_logit(evaluate, hess, initial, effective_weight, maxiter):
    """Scaled L-BFGS with gradient verification and damped Newton refinement.

    Scaling changes only optimizer coordinates, never data or likelihood.
    The starting coefficients are always a candidate, so a block M-step
    cannot decrease its expected complete-data log likelihood.
    """
    from scipy.optimize import OptimizeResult
    weight = max(float(effective_weight), 1.0)
    initial = np.array(initial, dtype=float, copy=True)
    curvature = np.maximum(np.diag(hess(np.zeros_like(initial))) / weight, 0.0)
    scale = np.sqrt(curvature)
    scale[scale < 1e-8] = 1.0
    calls = 0

    def objective(theta):
        nonlocal calls
        calls += 1
        value, gradient = evaluate(theta / scale)
        return value / weight, gradient / (weight * scale)

    theta0 = initial * scale
    f0, g0 = objective(theta0)
    best_theta, best_f = theta0.copy(), f0
    raw_message = 'Initial coefficients evaluated without optimization'
    nit = 0
    if maxiter > 0:
        res = minimize(objective, theta0, jac=True, method='L-BFGS-B', options={
            'maxiter': maxiter, 'maxfun': max(20000, 30 * maxiter),
            'ftol': OPT_FTOL, 'gtol': OPT_GTOL, 'maxls': 50, 'maxcor': 20,
        })
        raw_message, nit = str(res.message), int(res.nit)
        candidate_f, _ = objective(res.x)
        if np.isfinite(candidate_f) and candidate_f <= best_f:
            best_theta, best_f = res.x.copy(), candidate_f

    # A relative-function stop alone is insufficient on a massive likelihood.
    # Small dense Hessians (one logit block) make Newton polishing affordable.
    refinements = 0
    for _ in range(OPT_NEWTON_STEPS if maxiter > 0 else 0):
        value, grad = objective(best_theta)
        if np.linalg.norm(grad, ord=np.inf) <= OPT_GTOL:
            break
        H = hess(best_theta / scale) / weight / scale[:, None] / scale[None, :]
        jitter = 1e-10 * max(1.0, float(np.max(np.diag(H))))
        try:
            direction = -np.linalg.solve(H + jitter * np.eye(len(H)), grad)
        except np.linalg.LinAlgError:
            direction = -np.linalg.pinv(H) @ grad
        slope = float(grad @ direction)
        if not np.isfinite(direction).all() or slope >= 0:
            direction, slope = -grad, -float(grad @ grad)
        accepted = False
        for backtrack in range(40):
            step = 0.5 ** backtrack
            theta = best_theta + step * direction
            candidate_f, _ = objective(theta)
            if np.isfinite(candidate_f) and candidate_f <= value + 1e-4 * step * slope:
                best_theta, best_f = theta, candidate_f
                accepted = True
                refinements += 1
                break
        if not accepted:
            break
    # Recheck unscaled loss, preventing a rounding artifact in division from
    # accepting an objectively worse solution than the warm start.
    beta = best_theta / scale
    initial_value, _ = evaluate(initial)
    final_value, final_gradient = evaluate(beta)
    reverted = not np.isfinite(final_value) or final_value > initial_value
    if reverted:
        beta = initial
        final_value, final_gradient = evaluate(beta)
    scaled_gradient = final_gradient / (weight * scale)
    score = float(np.linalg.norm(scaled_gradient, ord=np.inf))
    success = bool(np.isfinite(score) and score <= OPT_GTOL)
    return OptimizeResult(
        x=beta, fun=final_value, jac=final_gradient, success=success,
        status=0 if success else 2, nit=nit + refinements, nfev=calls, njev=calls,
        message=('Scaled gradient tolerance met' if success else
                 'Scaled gradient tolerance NOT met; inspect this block'),
        scaled_gradient_inf=score, initial_objective=initial_value,
        objective_improvement=initial_value-final_value, reverted_to_start=reverted,
        lbfgs_message=raw_message, newton_refinements=refinements,
    )




def fit_weighted_logit(
    X, y, sample_weight=None, start=None, param_names=None, maxiter=300,
    compute_se=False, block_name="weighted_logit",
):
    """Weighted Bernoulli likelihood, evaluated once per optimizer candidate."""
    y = np.asarray(y, dtype=np.float64)
    weight = np.ones_like(y) if sample_weight is None else np.asarray(sample_weight, dtype=np.float64)
    if np.any(np.isfinite(weight) & (weight < 0)):
        raise ValueError(f"{block_name}: negative weights are not allowed.")
    names = list(param_names) if param_names is not None else [f"b{i}" for i in range(X.shape[1])]
    return fit_binomial_count_logit(
        X, weight*y, weight*(1-y), start, names, block_name, maxiter, compute_se,
    )


def fit_compressed_gate_logit(
    X_base, y, weight_state1, weight_state2, start, param_names, block_name,
    maxiter=300, compute_se=False,
):
    """Compatibility wrapper for the original two-state weighted-trip API."""
    y = np.asarray(y, dtype=np.float64)
    w1 = np.asarray(weight_state1, dtype=np.float64)
    w2 = np.asarray(weight_state2, dtype=np.float64)
    return fit_gate_count_logit(
        X_base, w1*y, w1*(1-y), w2*y, w2*(1-y), start,
        param_names, block_name, maxiter, compute_se,
    )


def fit_gate_count_logit(
    X_base, success1, failure1, success2, failure2, start,
    param_names, block_name, maxiter=300, compute_se=False,
):
    """Exact two-state binomial likelihood on unique, unrounded covariates.

    Counts already include annual posterior weights and the category weight.
    No binomial combinatorial constant is added (the original trip likelihood).
    """
    X_base = np.asarray(X_base, dtype=np.float64)
    counts = np.column_stack([success1, failure1, success2, failure2]).astype(np.float64, copy=False)
    if np.any(np.isfinite(counts) & (counts < 0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    keep = np.isfinite(counts).all(axis=1) & (counts.sum(axis=1)>0) & np.isfinite(X_base).all(axis=1)
    if not keep.all():
        X_base, counts = X_base[keep], counts[keep]
    if not len(counts):
        raise ValueError(f"{block_name}: no valid observations remain.")
    s1, f1, s2, f2 = counts.T
    t1, t2 = s1+f1, s2+f2
    p_count = X_base.shape[1]+1
    base_positions = np.r_[0, np.arange(2, p_count)]
    initial = np.zeros(p_count) if start is None else np.asarray(start, dtype=np.float64)

    def components(beta):
        shared = X_base @ beta[base_positions]
        return sigmoid(shared+beta[1]), sigmoid(shared+2*beta[1])

    def value_gradient(beta):
        p1, p2 = components(beta)
        shared = X_base @ beta[base_positions]
        eta1, eta2 = shared + beta[1], shared + 2 * beta[1]
        # Stable Bernoulli loss, exactly consistent with the analytic gradient.
        value = np.sum(s1*np.logaddexp(0, -eta1) + f1*np.logaddexp(0, eta1)
                       + s2*np.logaddexp(0, -eta2) + f2*np.logaddexp(0, eta2))
        r1, r2 = t1*p1-s1, t2*p2-s2
        out = np.empty(p_count)
        out[base_positions] = X_base.T @ (r1+r2)
        out[1] = np.sum(r1+2*r2)
        return float(value), out
    evaluate = _cached_value_gradient(value_gradient)

    def hess(beta):
        p1, p2 = components(beta)
        c1, c2 = t1*p1*(1-p1), t2*p2*(1-p2)
        out = np.empty((p_count, p_count))
        out[np.ix_(base_positions, base_positions)] = X_base.T @ (X_base*(c1+c2)[:, None])
        cross = X_base.T @ (c1+2*c2)
        out[base_positions, 1] = cross
        out[1, base_positions] = cross
        out[1, 1] = np.sum(c1+4*c2)
        return out

    result = _minimize_logit(evaluate, hess, initial, float(counts.sum()), maxiter)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(counts), float(counts.sum()), compute_se,
        "scaled_lbfgs_newton_gate_logit",
    )


def fit_binomial_count_logit(
    X, success, failure, start, param_names, block_name, maxiter=300, compute_se=False,
):
    """Original binomial-count objective with a joint loss/gradient callback."""
    X = np.asarray(X, dtype=np.float64)
    success, failure = np.asarray(success, dtype=np.float64), np.asarray(failure, dtype=np.float64)
    if np.any(np.isfinite(success) & (success<0)) or np.any(np.isfinite(failure) & (failure<0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    total = success+failure
    keep = np.isfinite(success) & np.isfinite(failure) & (total>0) & np.isfinite(X).all(axis=1)
    if not keep.all():
        X, success, failure, total = X[keep], success[keep], failure[keep], total[keep]
    if not len(total):
        raise ValueError(f"{block_name}: no positive expected counts remain.")
    initial = np.zeros(X.shape[1]) if start is None else np.asarray(start, dtype=np.float64)

    def value_gradient(beta):
        eta = X @ beta
        p = sigmoid(eta)
        value = np.sum(success*np.logaddexp(0, -eta) + failure*np.logaddexp(0, eta))
        return float(value), X.T @ (total*p-success)
    evaluate = _cached_value_gradient(value_gradient)
    def hess(beta):
        p = sigmoid(X @ beta)
        return X.T @ (X*(total*p*(1-p))[:, None])

    result = _minimize_logit(evaluate, hess, initial, float(total.sum()), maxiter)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(total), float(total.sum()), compute_se,
        "scaled_lbfgs_newton_binomial_logit",
    )


def finalize_covariance(fit: Dict[str, Any]) -> None:
    """Evaluate a final M-step Hessian once, with its original E-step weights."""
    callback = fit['_hessian_function']
    hessian = np.asarray(callback(fit['coef']), dtype=np.float64)
    if not np.isfinite(hessian).all():
        raise ValueError('Nonfinite final block Hessian.')
    hessian = (hessian + hessian.T)/2
    # An indefinite beta Hessian is not a valid covariance matrix.
    rank = int(np.linalg.matrix_rank(hessian))
    positive = bool(np.linalg.eigvalsh(hessian).min() > 0)
    valid = rank == len(hessian) and positive and not fit['diagnostics'].get('precision_at_bound', False)
    covariance = np.linalg.inv(hessian) if valid else np.full_like(hessian, np.nan)
    se = np.sqrt(np.clip(np.diag(covariance), 0.0, np.inf))
    with np.errstate(divide='ignore', invalid='ignore'):
        z = fit['coef'] / se
    table = fit['table']
    table['std_error'] = se
    table['z_value'] = z
    table['p_value'] = 2 * norm.sf(np.abs(z))
    table['significance_stars'] = table['p_value'].map(significance_stars)
    fit['hessian'], fit['cov'] = hessian, covariance
    fit['diagnostics']['covariance_hessian_success'] = bool(np.isfinite(covariance).all())
    fit['diagnostics']['hessian_rank'] = int(np.linalg.matrix_rank(hessian))
    fit['diagnostics']['parameter_count'] = len(se)


def beta_terms(beta, X):
    """Beta mean, derivatives and shape parameters; X includes the state column."""
    z = expit(X @ beta[:-1])
    mu = BETA_MEAN_EPS + (1-2*BETA_MEAN_EPS)*z
    dmu = (1-2*BETA_MEAN_EPS)*z*(1-z)
    d2mu = dmu*(1-2*z)
    phi = np.exp(beta[-1])
    return mu, dmu, d2mu, phi, mu*phi, (1-mu)*phi


def beta_objective(beta, X, count, log_y, log_1my, need_hessian=False):
    """Exact negative beta log density from n, sum(log y), sum(log(1-y))."""
    mu, dm, d2m, phi, a, b = beta_terms(beta, X)
    U = count*(digamma(a)-digamma(b))-log_y+log_1my
    V = count*(mu*digamma(a)+(1-mu)*digamma(b)-digamma(phi))-mu*log_y-(1-mu)*log_1my
    value = np.sum(count*betaln(a,b)-(a-1)*log_y-(b-1)*log_1my)
    grad = np.r_[X.T@(phi*U*dm), phi*V.sum()]
    if not need_hessian:
        return float(value), grad
    ta,tb,tp=polygamma(1,a),polygamma(1,b),polygamma(1,phi)
    hee = count*phi**2*(ta+tb)*dm**2+phi*U*d2m
    her = phi*dm*(U+phi*count*(mu*ta-(1-mu)*tb))
    hrr = phi*V+phi**2*count*(mu**2*ta+(1-mu)**2*tb-tp)
    H=np.empty((len(beta),len(beta)))
    H[:-1,:-1]=X.T@(X*hee[:,None]); H[:-1,-1]=X.T@her
    H[-1,:-1]=H[:-1,-1]; H[-1,-1]=hrr.sum()
    return H


def fit_beta_block(X, count, log_y, log_1my, start, names, block_name):
    """Scale optimizer coordinates; precision bounds protect numerical evaluation."""
    from scipy.optimize import OptimizeResult
    keep=count>0
    X,count,log_y,log_1my=X[keep],count[keep],log_y[keep],log_1my[keep]
    weight=float(count.sum())
    initial=np.zeros(X.shape[1]+1) if start is None else np.array(start,copy=True)
    if start is None: initial[-1]=np.log(5.)
    evaluate=_cached_value_gradient(lambda b: beta_objective(b,X,count,log_y,log_1my))
    hess=lambda b: beta_objective(b,X,count,log_y,log_1my,True)
    scale=np.r_[np.sqrt(np.average(X**2,axis=0,weights=count)),1.]
    scale[scale<1e-8]=1.
    def objective(theta):
        val,grad=evaluate(theta/scale)
        return val/weight, grad/(weight*scale)
    bounds=[(None,None)]*X.shape[1]+[LOG_PRECISION_BOUNDS]
    theta0=initial*scale; f0,_=objective(theta0)
    raw=minimize(objective,theta0,jac=True,method='L-BFGS-B',bounds=bounds,
        options=dict(maxiter=500,maxls=50,ftol=OPT_FTOL,gtol=OPT_GTOL,maxcor=20))
    theta=raw.x.copy() if np.isfinite(raw.fun) and raw.fun<=f0 else theta0.copy()
    refinements=0
    for _ in range(OPT_NEWTON_STEPS):
        val,grad=objective(theta)
        if np.max(np.abs(grad))<=OPT_GTOL: break
        H=hess(theta/scale)/weight/scale[:,None]/scale[None,:]
        try: direction=-np.linalg.solve(H+1e-10*np.eye(len(H)),grad)
        except np.linalg.LinAlgError: direction=-grad
        if not np.isfinite(direction).all() or grad@direction>=0: direction=-grad
        accepted=False
        for bt in range(40):
            candidate=theta+(.5**bt)*direction
            candidate[-1]=np.clip(candidate[-1],*LOG_PRECISION_BOUNDS)
            f,_=objective(candidate)
            if np.isfinite(f) and f<=val+1e-4*(grad@(candidate-theta)):
                theta=candidate; refinements+=1; accepted=True; break
        if not accepted: break
    value,grad=evaluate(theta/scale)
    score=float(np.max(np.abs(grad/(weight*scale))))
    at_bound=bool(min(abs(theta[-1]-v) for v in LOG_PRECISION_BOUNDS)<1e-6)
    success=bool(score<=OPT_GTOL and not at_bound)
    result=OptimizeResult(x=theta/scale,success=success,status=0 if success else 2,
        message='Scaled gradient met' if success else 'Inspect beta gradient / precision bound',
        nit=raw.nit+refinements,nfev=raw.nfev,njev=raw.njev,scaled_gradient_inf=score,
        initial_objective=f0*weight,objective_improvement=f0*weight-value,
        reverted_to_start=np.array_equal(theta,theta0),lbfgs_message=str(raw.message),newton_refinements=refinements)
    fit=_finish_logit_fit(result,hess,lambda b:evaluate(b)[1],names,block_name,
        lambda b:evaluate(b)[0],len(count),weight,False,'scaled_beta_lbfgs_newton')
    fit['diagnostics']['precision_at_bound']=at_bound
    return fit


## 2. Food-spending validation and exact annual statistics
Shares use `food_spend` for every category. Zero-food trips remain in the annual panel but have no share observation. The audit records denominator eligibility, zero/interior/one counts and absent-one component handling. No raw trip arrays enter EM.


In [ ]:
def sql_name(name):
    return '"'+name.replace('"','""')+'"'


def prepare_parquet(path, scratch_dir):
    """Validate first, then aggregate; never silently drop malformed records."""
    con=duckdb.connect()
    con.execute(f"SET threads={HMM_NUM_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    scratch_dir=Path(scratch_dir); scratch_dir.mkdir(parents=True,exist_ok=True)
    con.execute("SET temp_directory='"+str(scratch_dir).replace("'","''")+"'")
    con.from_parquet(str(path)).create_view('raw')
    controls=[*GATE_COVARIATE_COLS,*TRANSITION_EVENT_COLUMNS,FRESH_SHARE_2007_COL]
    spends=[cfg['healthy'] for cfg in CATEGORY_CONFIG.values()]
    required=['hhid','panel_year','trip_id','purchase_date','food_spend',*controls,*spends]
    actual={r[0] for r in con.execute('DESCRIBE raw').fetchall()}
    if set(required)-actual: raise ValueError(f'Missing columns: {set(required)-actual}')
    bad=[f'{sql_name(c)} IS NULL' for c in required]
    numeric=['panel_year','food_spend',*controls,*spends]
    bad += [f'NOT isfinite(CAST({sql_name(c)} AS DOUBLE))' for c in numeric]
    bad += ['panel_year NOT BETWEEN 2008 AND 2017','panel_year != floor(panel_year)',"TRY_CAST(purchase_date AS DATE) IS NULL"]
    bad += [f'{c} NOT IN (0,1)' for c in [*TRANSITION_EVENT_COLUMNS,'K_1']]
    bad += [f'{sql_name(c)} < 0' for c in ['food_spend',*spends]]
    bad += [f'(CAST({sql_name(c)} AS DOUBLE) - CAST(food_spend AS DOUBLE)) > {SPENDING_ATOL!r}' for c in spends]
    bad += ['family_size_numeric <= 0','age <= 0','age > 120','income10k_imputed < 0']
    n,b=con.execute('SELECT count(*), count(*) FILTER (WHERE '+ ' OR '.join(bad)+') FROM raw').fetchone()
    if b: raise ValueError(f'{b:,} invalid rows; require finite nonnegative spending, healthy spend - food_spend <= SPENDING_ATOL, and valid identifiers/demographics; fix upstream rather than clipping or filtering.')
    low,high=con.execute(f'SELECT min({FRESH_SHARE_2007_COL}),max({FRESH_SHARE_2007_COL}) FROM raw').fetchone()
    if not n or not (0<low<1) or abs(high-low)>1e-12: raise ValueError('Initial share must be one common value strictly between zero and one.')
    if CHECK_DUPLICATE_TRIPS:
        duplicate=con.execute('SELECT 1 FROM raw GROUP BY hhid,panel_year,trip_id HAVING count(*)>1 LIMIT 1').fetchone()
        if duplicate: raise ValueError('Duplicate household-year-trip key.')
    checks=' OR '.join(f'min({c}) != max({c})' for c in controls)
    if con.execute(f'SELECT 1 FROM raw GROUP BY hhid,panel_year HAVING {checks} LIMIT 1').fetchone():
        raise ValueError('Demographics/events/initial share vary within a household-year.')
    con.execute(f'CREATE TEMP TABLE eligible AS SELECT hhid FROM raw GROUP BY hhid HAVING count(DISTINCT panel_year)>={MIN_OBSERVED_YEARS_PER_HOUSEHOLD}')
    # Audit raw discrepancies; all subsequent statistics use the same corrected numerator.
    audit_parts=[]
    for category,cfg in CATEGORY_CONFIG.items():
        h=f'CAST({sql_name(cfg["healthy"])} AS DOUBLE)'
        excess=f'({h}-CAST(food_spend AS DOUBLE))'
        audit_parts.append(f"SELECT '{category}' AS category, count(*) FILTER(WHERE {excess}>0 AND {excess}<={SPENDING_ATOL!r}) AS corrected_rows, max({excess}) FILTER(WHERE {excess}>0) AS maximum_raw_excess FROM raw")
    con.execute(' UNION ALL '.join(audit_parts)).df().to_csv(scratch_dir.parent/'floating_point_spending_audit.csv',index=False)
    expressions=[]
    for c,cfg in CATEGORY_CONFIG.items():
        raw_healthy=f'CAST({sql_name(cfg["healthy"])} AS DOUBLE)'
        healthy=f'(CASE WHEN {raw_healthy}>CAST(food_spend AS DOUBLE) THEN CAST(food_spend AS DOUBLE) ELSE {raw_healthy} END)'
        expressions += [f'CAST(food_spend AS DOUBLE) AS {c}_total',
            f'{healthy} AS {c}_healthy', f'(CAST(food_spend AS DOUBLE)-{healthy}) AS {c}_other']
    con.execute('CREATE VIEW amounts AS SELECT r.*,'+','.join(expressions)+' FROM raw r SEMI JOIN eligible e USING(hhid)')
    expressions=[]
    for c in CATEGORY_NAMES:
        expressions += [f'{c}_healthy/NULLIF({c}_total,0) AS {c}_share',
            f'({c}_healthy>0 AND {c}_other>0) AS {c}_interior',
            f'({c}_healthy>0 AND {c}_other=0) AS {c}_one',
            f'({c}_healthy=0 AND {c}_other>0) AS {c}_zero']
    con.execute('CREATE VIEW trips AS SELECT *,'+','.join(expressions)+' FROM amounts')
    # Boundary membership uses the original amounts, not rounded ratios.
    bad_total=' OR '.join(f'NOT isfinite({c}_total)' for c in CATEGORY_NAMES)
    if con.execute('SELECT count(*) FROM trips WHERE '+bad_total).fetchone()[0]:
        raise ValueError('Overflow in category spending totals.')
    # SQL aliases retain the internal V29 household key to reuse its tested recursion.
    aggs=['CAST(hhid AS VARCHAR) AS household_code','CAST(panel_year AS INTEGER) AS panel_year','count(*) AS n_trips',
          'sum(CAST(food_spend AS DOUBLE)) AS food_total_spend_sum',
          'count(*) FILTER(WHERE food_spend>0) AS food_positive_n',
          'min(CAST(purchase_date AS DATE)) AS first_purchase_date','max(CAST(purchase_date AS DATE)) AS last_purchase_date']
    aggs += [f'first(CAST({c} AS DOUBLE)) AS {c}' for c in controls]
    for c in CATEGORY_NAMES:
        aggs += [f'count(*) FILTER(WHERE {c}_total>0) AS {c}_eligible_n',
            f'sum({c}_total) AS {c}_spend_sum', f'sum({c}_healthy) AS {c}_healthy_spend_sum',
            f'coalesce(sum({c}_share),0) AS {c}_share_sum',
            f'count(*) FILTER(WHERE {c}_interior) AS {c}_interior_n',
            f'count(*) FILTER(WHERE {c}_one) AS {c}_one_n',
            f'count(*) FILTER(WHERE {c}_zero) AS {c}_zero_n',
            # Difference of logs avoids rounding an interior ratio to 1.
            f'sum(CASE WHEN {c}_interior THEN ln({c}_healthy)-ln({c}_total) ELSE 0 END) AS {c}_log_y_sum',
            f'sum(CASE WHEN {c}_interior THEN ln({c}_other)-ln({c}_total) ELSE 0 END) AS {c}_log_1my_sum']
    hy=con.execute('SELECT '+','.join(aggs)+' FROM trips GROUP BY hhid,panel_year ORDER BY household_code,panel_year').df()
    if hy.empty: raise ValueError('No households pass the observed-year filter.')
    hh=hy.household_code.to_numpy(); starts=np.flatnonzero(np.r_[True,hh[1:]!=hh[:-1]])
    ends=np.r_[starts[1:],len(hy)]
    gaps=(hy.groupby('household_code',sort=False).panel_year.diff()>1)
    if YEAR_GAP_POLICY not in ('observed_step','error'): raise ValueError('Invalid YEAR_GAP_POLICY')
    if gaps.any() and YEAR_GAP_POLICY=='error': raise ValueError(f'{gaps.sum()} observed-year gaps; explicit handling required.')
    hy['chain_step']=hy.groupby('household_code',sort=False).cumcount()+1
    x=np.column_stack([np.ones(len(hy)),hy[TRANSITION_EVENT_COLUMNS].to_numpy(float)])
    mask=np.ones(len(hy),bool); mask[starts]=False
    curr=np.flatnonzero(mask); row=np.full(len(hy),-1,dtype=np.int64); row[curr]=np.arange(len(curr))
    annual_cols=['household_code','panel_year','n_trips','food_total_spend_sum',*TRANSITION_EVENT_COLUMNS]
    annual_cols += [c for c in hy if c.endswith(('_n','_sum')) and c not in annual_cols]
    annual=hy[annual_cols].copy()
    base=np.column_stack([np.ones(len(hy)),hy[GATE_COVARIATE_COLS].to_numpy(float)])
    unique,mapping=np.unique(base,axis=0,return_inverse=True)
    blocks={}; audit=[]
    for c in CATEGORY_NAMES:
        purchase_count=hy[c+'_eligible_n'].to_numpy(float)
        interior=hy[c+'_interior_n'].to_numpy(float)
        one=hy[c+'_one_n'].to_numpy(float); zero=hy[c+'_zero_n'].to_numpy(float)
        audit.append(dict(category=c,no_spending=int(hy.n_trips.sum()-purchase_count.sum()),
            zero_share=int(zero.sum()),interior_share=int(interior.sum()),one_share=int(one.sum()),
            healthy_numerator=CATEGORY_CONFIG[c]['healthy'], denominator='food_spend',
            boundary_one_mode='estimated' if one.sum()>0 else 'fixed_zero_no_ones'))
        for part,success,total in [('interior',interior,purchase_count),('boundary_one',one,one+zero)]:
            if part=='boundary_one' and one.sum()==0: continue
            active=np.flatnonzero(total>0); key=c+'_'+part
            blocks[key]=GateBlock(key,c,key,unique,success[active]/total[active],active,int(total.sum()),1.,
                success[active],total[active]-success[active],mapping[active])
    boundary_audit=pd.DataFrame(audit)
    (scratch_dir.parent).mkdir(parents=True,exist_ok=True)
    boundary_audit.to_csv(scratch_dir.parent/'share_data_audit.csv',index=False)
    absent=boundary_audit.loc[(boundary_audit[['zero_share','interior_share']]==0).any(axis=1),'category'].tolist()
    if absent:
        con.close()
        raise ValueError(f'V36 share specification is unidentified for {absent}: a zero or interior component is absent. See share_data_audit.csv; do not manufacture fractions. Simplify the affected category model explicitly.')
    arrays=ModelArrays({}, {}, {}, np.array([]),np.array([]),{},blocks,np.array([],dtype=int),hy,
        pd.MultiIndex.from_frame(hy[['household_code','panel_year']]),{},hh[starts].tolist(),{},starts,ends,x,curr,row)
    arrays._share_design=unique
    arrays._share_design_pos=mapping
    arrays._share_audit=boundary_audit
    arrays._connection=con
    arrays._annual_statistics=annual
    shell=pd.DataFrame(index=pd.RangeIndex(int(hy.n_trips.sum())))
    shell.attrs['sorting_audit']={'input_trips':n,'retained_trips':len(shell),'minimum_observed_years_required':MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        'trips_removed_by_minimum_years_filter':n-len(shell),'observed_year_gaps':int(gaps.sum()),'year_gap_policy':YEAR_GAP_POLICY,
        'positive_food_spending_trips':int(hy.food_positive_n.sum()),
        'zero_food_spending_trips_neutral_emission':int(hy.n_trips.sum()-hy.food_positive_n.sum()),
        'household_years_without_share_observations':int((hy.food_positive_n==0).sum())}
    print(f'Input {n:,} trips; retained {len(shell):,}; {len(starts):,} households; {len(hy):,} household-years; {gaps.sum():,} gaps.')
    return PreparedData(shell,hy,arrays,annual)


def initial_gate_weights(prepared):
    """Non-identical emission starts, independent of the constant external pi.

    Smoothed annual mean trip fresh/food spending share supplies a weak initial ordering. Its only role
    is numerical initialization; subsequent posterior weights come from the HMM.
    """
    hy=prepared.hy
    score=(hy.fv_share_sum.to_numpy(float)+.5)/(hy.food_positive_n.to_numpy(float)+1.)
    z=(score-score.mean())/max(score.std(),1e-8)
    g=expit(.7*np.clip(z,-3,3))
    out=hy[['household_code','panel_year']].copy()
    out['gamma_state_1']=1-g; out['gamma_state_2']=g
    return out


## 3. HMM and no-switch estimation
All six share emissions use every positive-food-spending trip. The former category-incidence terms are absent. Category boundary-one gates are fitted only when 100% shares exist; the corresponding probabilities are fixed at zero otherwise. The annual forward–backward recursion and safeguarded EM remain unchanged.


In [ ]:
def _align_gamma_to_hy(gamma_df: pd.DataFrame, arrays: ModelArrays) -> Tuple[np.ndarray, np.ndarray]:
    """Use zero-copy aligned posterior fast path, retaining a safe keyed fallback."""
    if len(gamma_df) == len(arrays.household_year):
        same_hh = np.array_equal(
            gamma_df["household_code"].astype(str).to_numpy(),
            arrays.household_year["household_code"].astype(str).to_numpy(),
        )
        same_year = np.array_equal(
            gamma_df["panel_year"].to_numpy(dtype=np.int64),
            arrays.household_year["panel_year"].to_numpy(dtype=np.int64),
        )
        if same_hh and same_year:
            return (
                gamma_df["gamma_state_1"].to_numpy(dtype=np.float64, copy=False),
                gamma_df["gamma_state_2"].to_numpy(dtype=np.float64, copy=False),
            )
    gamma_key = pd.MultiIndex.from_frame(gamma_df[["household_code", "panel_year"]])
    aligned = gamma_df.set_index(gamma_key).reindex(arrays.hy_key)
    if aligned[["gamma_state_1", "gamma_state_2"]].isna().any().any():
        raise ValueError("gamma_df cannot be aligned to the household-year index.")
    return aligned["gamma_state_1"].to_numpy(dtype=np.float64), aligned["gamma_state_2"].to_numpy(dtype=np.float64)

def fit_gate_models(
    arrays: ModelArrays,
    gamma_df: pd.DataFrame,
    progress_bar: Optional[Any] = None,
    em_iteration: Optional[int] = None,
    max_em_iters: Optional[int] = None,
    previous_fits: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    """Fit cached compressed gate likelihoods with one weight pair per eligible row."""
    gamma1, gamma2 = _align_gamma_to_hy(gamma_df, arrays)
    fitted: Dict[str, Any] = {}
    for gate_key in arrays.gate_blocks:
        block = arrays.gate_blocks[gate_key]
        if progress_bar is not None:
            if em_iteration is None:
                progress_bar.set_description_str(f"Initialization: {progress_bar.n + 1}/{progress_bar.total} | gate={gate_key}")
            else:
                progress_bar.set_description_str(f"EM {em_iteration}/{max_em_iters} | gate={gate_key}")
        # Sum fractional successes/failures over annual groups with the SAME
        # exact design. Four small count vectors replace all trip-state rows.
        n_design = len(block.x_base)
        w1 = gamma1[block.hy_pos]*block.choice_weight
        w2 = gamma2[block.hy_pos]*block.choice_weight
        counts = [np.bincount(block.design_pos, weights=w*count, minlength=n_design)
                  for w,count in ((w1,block.success_count), (w1,block.failure_count),
                                  (w2,block.success_count), (w2,block.failure_count))]
        start = previous_fits[gate_key]["coef"] if previous_fits is not None else None
        fitted[gate_key] = fit_gate_count_logit(
            block.x_base, *counts, start=start,
            param_names=[f"{block.prefix}_{col}" for col in GATE_X_COLS],
            block_name=f"gate:{gate_key}",
        )
        fitted[gate_key]["diagnostics"]["compressed_design_rows"] = n_design
        fitted[gate_key]["diagnostics"]["n_observations"] = block.eligible_rows
        if progress_bar is not None and em_iteration is None:
            progress_bar.update(1)
    # Beta sufficient statistics aggregate exactly over identical covariates.
    U=arrays._share_design; mapping=arrays._share_design_pos
    X=np.vstack([np.column_stack([U[:,0],np.full(len(U),s),U[:,1:]]) for s in (1.,2.)])
    hy=arrays.household_year
    for c in CATEGORY_NAMES:
        key=c+'_beta'; grouped=[]
        for field in ('interior_n','log_y_sum','log_1my_sum'):
            v=hy[c+'_'+field].to_numpy(float)
            grouped.append(np.concatenate([np.bincount(mapping,weights=g*v,minlength=len(U)) for g in (gamma1,gamma2)]))
        start=None if previous_fits is None else previous_fits[key]['coef']
        fitted[key]=fit_beta_block(X,*grouped,start,
            [f'{c}_beta_mean_{v}' for v in GATE_X_COLS]+[f'{c}_log_precision'],key)
        fitted[key]['diagnostics']['n_observations']=int(hy[c+'_interior_n'].sum())
    return fitted

def log_emission_by_household_year(arrays, incidence_models, gate_params):
    """Evaluate the exact trip share likelihood from annual sufficient statistics."""
    n_hy = len(arrays.household_year)
    common = np.zeros(n_hy)
    state1, state2 = common.copy(), common.copy()
    for key in arrays.gate_blocks:
        block = arrays.gate_blocks[key]
        coef = np.asarray(gate_params[key]["coef"], dtype=np.float64)
        shared = block.x_base @ np.r_[coef[0], coef[2:]]
        for state_num, destination in ((1.0,state1), (2.0,state2)):
            eta = shared + state_num * coef[1]
            lp, lq = -np.logaddexp(0, -eta), -np.logaddexp(0, eta)
            ll = (block.success_count*lp[block.design_pos]
                  +block.failure_count*lq[block.design_pos])
            # hy_pos is unique in a gate, so direct indexed addition is valid.
            destination[block.hy_pos] += block.choice_weight*ll
    U=arrays._share_design; mapping=arrays._share_design_pos; hy=arrays.household_year
    for c in CATEGORY_NAMES:
        count=hy[c+'_interior_n'].to_numpy(float)
        ly=hy[c+'_log_y_sum'].to_numpy(float); lq=hy[c+'_log_1my_sum'].to_numpy(float)
        for s,destination in ((1.,state1),(2.,state2)):
            X=np.column_stack([U[:,0],np.full(len(U),s),U[:,1:]])
            _,_,_,_,a,b=beta_terms(gate_params[c+'_beta']['coef'],X)
            a,b=a[mapping],b[mapping]
            destination += (a-1)*ly+(b-1)*lq-count*betaln(a,b)
    return np.column_stack([state1,state2])

def forward_backward(arrays: ModelArrays, emission_loglik: np.ndarray, pi_state2: float,
                     beta_from1: np.ndarray, beta_from2: np.ndarray):
    """Same log-space recursions, vectorized over equal-length households.

    Observed-year gaps count as one step only under the explicit observed_step policy.
    """
    n_hy = len(arrays.household_year)
    gamma_values = np.empty((n_hy, 2), dtype=np.float64)
    xi_values = np.empty((len(arrays.transition_curr_pos), 4), dtype=np.float64)
    eta1 = arrays.transition_x @ beta_from1
    eta2 = arrays.transition_x @ beta_from2
    log_trans = np.stack([-np.logaddexp(0, eta1), -np.logaddexp(0, -eta1),
                          -np.logaddexp(0, eta2), -np.logaddexp(0, -eta2)], axis=1).reshape(n_hy, 2, 2)
    log_pi = np.array([safe_log(1-pi_state2), safe_log(pi_state2)])
    if not hasattr(arrays, '_sequence_batches'):
        lengths = arrays.hh_ends - arrays.hh_starts
        arrays._sequence_batches = [(np.flatnonzero(lengths == t),
            arrays.hh_starts[lengths == t, None] + np.arange(t)) for t in np.unique(lengths)]
    household_lls = np.empty(len(arrays.hh_starts))
    for hh_indices, pos in arrays._sequence_batches:
        em = emission_loglik[pos]
        trans = log_trans[pos]
        alpha = np.zeros_like(em); backward = np.zeros_like(em)
        alpha[:, 0] = log_pi + em[:, 0]
        for t in range(1, pos.shape[1]):
            alpha[:, t, 0] = em[:, t, 0]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 0], alpha[:, t-1, 1]+trans[:, t, 1, 0])
            alpha[:, t, 1] = em[:, t, 1]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 1], alpha[:, t-1, 1]+trans[:, t, 1, 1])
        ll = np.logaddexp(alpha[:, -1, 0], alpha[:, -1, 1])
        household_lls[hh_indices] = ll
        for t in range(pos.shape[1]-2, -1, -1):
            nxt = em[:, t+1]+backward[:, t+1]
            backward[:, t, 0] = np.logaddexp(trans[:, t+1, 0, 0]+nxt[:, 0], trans[:, t+1, 0, 1]+nxt[:, 1])
            backward[:, t, 1] = np.logaddexp(trans[:, t+1, 1, 0]+nxt[:, 0], trans[:, t+1, 1, 1]+nxt[:, 1])
        gamma = np.exp(alpha + backward - ll[:, None, None])
        gamma /= gamma.sum(axis=2, keepdims=True)
        gamma_values[pos] = gamma
        for t in range(1, pos.shape[1]):
            xi = np.exp(alpha[:, t-1, :, None] + trans[:, t] + em[:, t, None, :]
                        + backward[:, t, None, :] - ll[:, None, None])
            xi /= xi.sum(axis=(1, 2), keepdims=True)
            xi_values[arrays.transition_row_of_hy[pos[:, t]]] = xi.reshape(-1, 4)
    # Sum in original household order to minimize numerical changes.
    total_ll = sum(float(x) for x in household_lls)
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma_values.T
    gamma_df['decoded_state'] = np.where(gamma_values[:, 1] > gamma_values[:, 0], 2, 1)
    curr = arrays.transition_curr_pos
    xi_df = pd.DataFrame({'household_code': arrays.household_year['household_code'].to_numpy()[curr],
        'year_prev': arrays.household_year['panel_year'].to_numpy()[curr-1],
        'year_curr': arrays.household_year['panel_year'].to_numpy()[curr],
        **{c: xi_values[:, j] for j, c in enumerate(['xi_11', 'xi_12', 'xi_21', 'xi_22'])},
        **{c: arrays.transition_x[curr, j] for j, c in enumerate(TRANSITION_EVENT_COLUMNS, 1)}})
    return total_ll, gamma_df, xi_df

def fit_transition_models(
    xi_df: pd.DataFrame,
    start_from1: Optional[np.ndarray] = None,
    start_from2: Optional[np.ndarray] = None,
    arrays: Optional[ModelArrays] = None,
    maxiter: int = 300,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Fit both reference-preserving transition logits from compressed counts."""
    # Six binary flags imply at most 64 distinct designs. Cache their mapping.
    if arrays is not None and hasattr(arrays, "_transition_grouping"):
        X, codes = arrays._transition_grouping
    else:
        flags = np.column_stack([xi_df[c].to_numpy(dtype=np.int8) for c in TRANSITION_EVENT_COLUMNS])
        all_codes = flags @ (1 << np.arange(len(TRANSITION_EVENT_COLUMNS)))
        present, codes = np.unique(all_codes, return_inverse=True)
        X = np.column_stack([np.ones(len(present)),
                             ((present[:, None] >> np.arange(len(TRANSITION_EVENT_COLUMNS))) & 1)])
        codes = codes.astype(np.int32)
        if arrays is not None:
            arrays._transition_grouping = X, codes
    counts = {c:np.bincount(codes, weights=xi_df[c].to_numpy(), minlength=len(X))
              for c in ("xi_11", "xi_12", "xi_21", "xi_22")}
    fit1 = fit_binomial_count_logit(
        X, counts["xi_12"], counts["xi_11"], start_from1,
        [f"trans_from1_{col}" for col in TRANSITION_X_COLS], "transition:from_state_1", maxiter=maxiter,
    )
    fit2 = fit_binomial_count_logit(
        X, counts["xi_22"], counts["xi_21"], start_from2,
        [f"trans_from2_{col}" for col in TRANSITION_X_COLS], "transition:from_state_2", maxiter=maxiter,
    )
    for fit in (fit1,fit2):
        fit["diagnostics"]["compressed_design_rows"] = len(X)
        fit["diagnostics"]["n_observations"] = len(xi_df)
    return fit1, fit2

def _run_monotone_em(prepared, incidence, max_em_iters, tolerance, no_switch=False):
    """Shared EM policy: warm starts, observed-LL guard, persistent plateau.

    A rejected downhill proposal is logged, never relabeled as an improvement.
    Roundoff-sized rejections can count toward a stable plateau only when all
    candidate M-steps satisfy their gradient checks. Larger drops stop with
    converged=False and preserve the last accepted coefficients/posteriors.
    """
    if max_em_iters < 1 or tolerance <= 0:
        raise ValueError('Require max_em_iters >= 1 and tolerance > 0.')
    started_all = time.perf_counter()
    arrays = prepared.arrays
    gates = fit_gate_models(arrays, initial_gate_weights(prepared))
    pi = float(prepared.hy[FRESH_SHARE_2007_COL].iloc[0])
    beta1 = np.zeros(len(TRANSITION_X_COLS)); beta2 = beta1.copy()

    def e_step(g, p, b1, b2):
        emission = log_emission_by_household_year(arrays, incidence, g)
        if no_switch:
            ll, gamma = no_switch_e_step(arrays, emission, p)
            return ll, gamma, None
        return forward_backward(arrays, emission, p, b1, b2)

    ll, gamma, xi = e_step(gates, pi, beta1, beta2)
    trans = []
    if not no_switch:
        # Valid coefficient/covariance containers even if the FIRST EM step
        # is rejected. maxiter=0 evaluates the specified zero coefficients.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            trans = list(fit_transition_models(xi, beta1, beta2, arrays, maxiter=0))
    history = []
    stable = 0
    converged = False
    stop_reason = 'max_iterations_reached'
    label = 'No-switch' if no_switch else 'Proposed'
    incidence_ok = True  # No separate category-incidence equations in V36.
    for iteration in range(1, max_em_iters + 1):
        started = time.perf_counter()
        new_pi = pi  # Fixed external initial probability: no M-step update.
        new_trans = [] if no_switch else list(fit_transition_models(xi, beta1, beta2, arrays))
        new_beta1, new_beta2 = (beta1, beta2) if no_switch else tuple(f['coef'] for f in new_trans)
        new_gates = fit_gate_models(arrays, gamma, previous_fits=gates)
        candidate_ll, candidate_gamma, candidate_xi = e_step(new_gates, new_pi, new_beta1, new_beta2)
        candidate_delta = candidate_ll - ll
        failures = sum(not f['result'].success for f in [*new_trans, *new_gates.values()])
        threshold = tolerance + EM_REL_TOL * max(1.0, abs(ll))
        # Tolerance for arithmetic noise, much smaller than an EM stop threshold.
        roundoff = max(1e-8, 64 * np.finfo(float).eps * max(1.0, abs(ll)))
        accepted = bool(np.isfinite(candidate_ll) and candidate_delta >= 0)
        material_drop = not np.isfinite(candidate_ll) or candidate_delta < -roundoff
        delta = 0.0
        if accepted:
            delta = candidate_delta
            pi, gates, trans = new_pi, new_gates, new_trans
            beta1, beta2 = new_beta1, new_beta2
            ll, gamma, xi = candidate_ll, candidate_gamma, candidate_xi
        plateau = not material_drop and delta <= threshold and failures == 0 and incidence_ok
        stable = stable + 1 if plateau else 0
        converged = stable >= EM_STABLE_ITERS
        if material_drop:
            stop_reason = 'downhill_step_rejected'
        elif converged:
            stop_reason = 'stable_likelihood_and_block_gradients'
        history.append({
            'phase': 'EM', 'iteration': iteration, 'log_likelihood': ll,
            'log_likelihood_change': delta, 'candidate_log_likelihood_change': candidate_delta,
            'step_accepted': accepted, 'likelihood_parameter_stage': 'after_M_step',
            'iteration_seconds': time.perf_counter()-started,
            'elapsed_seconds': time.perf_counter()-started_all,
            'converged': converged, 'tolerance': tolerance, 'relative_tolerance': EM_REL_TOL,
            'effective_ll_tolerance': threshold, 'stable_iterations': stable,
            'max_em_iters': max_em_iters, 'optimizer_failures': failures,
            'stop_reason': stop_reason,
        })
        if iteration == 1 or iteration % EM_PRINT_EVERY == 0 or converged or material_drop or iteration == max_em_iters:
            print(f'{label} EM {iteration:03d}/{max_em_iters}: LL={ll:.6f}, '
                  f'delta={delta:.6g}, candidate_delta={candidate_delta:.6g}, '
                  f'block_failures={failures}, stable={stable}/{EM_STABLE_ITERS}')
        if material_drop:
            warnings.warn(f'{label}: downhill EM proposal rejected; last accepted fit retained.', RuntimeWarning)
            break
        if converged:
            break
    return dict(gates=gates, pi=pi, beta1=beta1, beta2=beta2, trans=trans,
                ll=ll, gamma=gamma, history=pd.DataFrame(history), converged=converged,
                stop_reason=stop_reason, estimation_seconds=time.perf_counter()-started_all)

def estimate_model(prepared: PreparedData, max_em_iters: int = MAX_EM_ITERS,
                   tolerance: float = TOL) -> EstimationResult:
    started_all = time.perf_counter()
    print('Fitting six healthy/food-spending emissions; safeguarded proposed-model EM...')
    prepared.arrays._connection.close()
    del prepared.arrays._connection
    incidence = {}  # Legacy result field retained for compatibility; no likelihood terms.

    out = _run_monotone_em(prepared, incidence, max_em_iters, tolerance)
    for f in [*out['trans'], *out['gates'].values()]:
        finalize_covariance(f)
    initial_table = pd.DataFrame()  # Fixed pi has no estimated coefficient.
    trans1, trans2 = out['trans']
    result = EstimationResult(prepared, incidence, out['gates'], out['beta1'], out['beta2'], out['pi'],
        out['gamma'], out['history'], initial_table,
        pd.concat([trans1['table'], trans2['table']], ignore_index=True), out['ll'],
        {'from_state_1': trans1, 'from_state_2': trans2})
    result.optimizer_diagnostics_df = collect_optimizer_diagnostics(result)
    result.estimation_seconds = time.perf_counter()-started_all
    result.converged = out['converged']
    result.stop_reason = out['stop_reason']
    return result

def collect_optimizer_diagnostics(result: EstimationResult) -> pd.DataFrame:
    """Return one diagnostic row for every final optimized logit block."""
    rows = []
    if result.transition_fits is None:
        raise RuntimeError("Final transition fits are unavailable for optimizer diagnostics.")
    rows.extend([
        result.transition_fits["from_state_1"]["diagnostics"],
        result.transition_fits["from_state_2"]["diagnostics"],
    ])
    rows.extend(fit["diagnostics"] for fit in result.gate_params.values())
    return pd.DataFrame(rows)


In [ ]:
@dataclass
class NoSwitchResult:
    """Two latent household classes with exactly zero switching probabilities."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    pi_state2: float
    gamma_df: pd.DataFrame
    log_likelihood: float
    history_df: pd.DataFrame
    converged: bool
    estimation_seconds: float

def no_switch_e_step(arrays, emission_loglik, pi_state2):
    """Exact household mixture likelihood for P = [[1, 0], [0, 1]].

    Sum annual log emissions within each household BEFORE mixing the two
    classes. Each household has one latent class for its entire sequence.
    Direct mixture evaluation avoids clipping structural zeros to epsilon.
    """
    if not 0 < pi_state2 < 1:
        raise ValueError("Initial class probability must be strictly between 0 and 1.")
    household_emission = np.add.reduceat(emission_loglik, arrays.hh_starts, axis=0)
    joint = household_emission + np.array([np.log1p(-pi_state2), np.log(pi_state2)])
    household_ll = np.logaddexp(joint[:, 0], joint[:, 1])
    posterior = np.exp(joint - household_ll[:, None])
    posterior /= posterior.sum(axis=1, keepdims=True)
    # Contiguous annual groups use the identical posterior in every year.
    gamma = np.repeat(posterior, arrays.hh_ends - arrays.hh_starts, axis=0)
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma.T
    gamma_df['decoded_state'] = np.where(gamma[:, 1] > gamma[:, 0], 2, 1)
    return float(household_ll.sum()), gamma_df

def estimate_no_switch_model(proposed, max_em_iters=MAX_EM_ITERS, tolerance=TOL):
    """Re-estimate fixed household classes with the same safeguarded optimizer.

    Use the same share support and fixed mixing probability; emission parameters
    are independently fit. The incidence container is empty in V36.
    """
    out = _run_monotone_em(proposed.prepared, proposed.incidence_models,
                           max_em_iters, tolerance, no_switch=True)
    # Evaluate final nested share-block Hessians with posterior weights fixed.
    for fit in out['gates'].values():
        finalize_covariance(fit)
    result = NoSwitchResult(proposed.prepared, proposed.incidence_models, out['gates'],
        out['pi'], out['gamma'], out['ll'], out['history'], out['converged'], out['estimation_seconds'])
    result.stop_reason = out['stop_reason']
    return result


## 4. Posterior healthy-food shares, significance stars and exports
The main table tests State 2 minus State 1 mean trip shares; the secondary table tests pooled food-budget shares. Both include household-clustered SEs, difference confidence intervals, two-sided p-values, and significance stars. Fitted model-implied shares are separate point estimates.


In [ ]:
def build_annual_sufficient_statistics(data):
    return data.annual_stats.copy()

def build_transition_matrix(beta_from1: np.ndarray, beta_from2: np.ndarray, x: np.ndarray) -> pd.DataFrame:
    """Build a two-by-two transition matrix at a supplied annual covariate vector."""
    p12 = float(sigmoid(np.dot(beta_from1, x)))
    p22 = float(sigmoid(np.dot(beta_from2, x)))
    return pd.DataFrame({
        "from_state": ["State 1", "State 2"],
        "to_state_1": [1.0 - p12, 1.0 - p22],
        "to_state_2": [p12, p22],
    })

def build_transition_outputs(result: EstimationResult) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build baseline and event-specific transition matrices at mean covariates."""
    hy = result.prepared.hy
    mean_x = np.array([
        1.0 if col == "intercept" else float(hy[col].mean())
        for col in TRANSITION_X_COLS
    ], dtype=np.float64)
    baseline = build_transition_matrix(result.beta_from1, result.beta_from2, mean_x)
    rows: List[pd.DataFrame] = []
    for event in TRANSITION_EVENT_COLUMNS:
        position = TRANSITION_X_COLS.index(event)
        for event_value in [0.0, 1.0]:
            x = mean_x.copy()
            x[position] = event_value
            matrix = build_transition_matrix(result.beta_from1, result.beta_from2, x)
            matrix.insert(0, "event", event)
            matrix.insert(1, "event_value", int(event_value))
            rows.append(matrix)
    return baseline, pd.concat(rows, ignore_index=True)

def _centered_sum_of_squares(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=np.float64)
    total = float(values.sum())
    return max(float(np.dot(values, values)) - total * total / len(values), 0.0)

def infer_posterior_ratio_difference(
    household_stats: pd.DataFrame,
    numerator_stat: str,
    denominator_stat: str,
) -> Dict[str, Any]:
    """Household-clustered delta-method inference for State 2 minus State 1."""
    n_clusters = len(household_stats)
    if n_clusters < 2:
        raise ValueError("At least two households are required for clustered profile inference.")

    numerator_s1 = household_stats[f"{numerator_stat}_s1"].to_numpy(dtype=np.float64, copy=False)
    numerator_s2 = household_stats[f"{numerator_stat}_s2"].to_numpy(dtype=np.float64, copy=False)
    denominator_s1 = household_stats[f"{denominator_stat}_s1"].to_numpy(dtype=np.float64, copy=False)
    denominator_s2 = household_stats[f"{denominator_stat}_s2"].to_numpy(dtype=np.float64, copy=False)
    total_denominator_s1 = float(denominator_s1.sum())
    total_denominator_s2 = float(denominator_s2.sum())

    if total_denominator_s1 <= 0 or total_denominator_s2 <= 0:
        return {
            "state_1_estimate": np.nan,
            "state_1_standard_error": np.nan,
            "state_2_estimate": np.nan,
            "state_2_standard_error": np.nan,
            "difference_state2_minus_state1": np.nan,
            "difference_standard_error": np.nan,
            "z_value": np.nan,
            "p_value": np.nan,
            "significance_stars": "",
            "difference_with_stars": "",
            "inference": "not_available",
            "n_household_clusters": n_clusters,
        }

    state_1_estimate = float(numerator_s1.sum() / total_denominator_s1)
    state_2_estimate = float(numerator_s2.sum() / total_denominator_s2)
    influence_s1 = (numerator_s1 - state_1_estimate * denominator_s1) / total_denominator_s1
    influence_s2 = (numerator_s2 - state_2_estimate * denominator_s2) / total_denominator_s2
    influence_difference = influence_s2 - influence_s1
    finite_sample_correction = n_clusters / (n_clusters - 1.0)
    state_1_standard_error = math.sqrt(
        finite_sample_correction * _centered_sum_of_squares(influence_s1)
    )
    state_2_standard_error = math.sqrt(
        finite_sample_correction * _centered_sum_of_squares(influence_s2)
    )
    difference_standard_error = math.sqrt(
        finite_sample_correction * _centered_sum_of_squares(influence_difference)
    )
    difference = state_2_estimate - state_1_estimate

    if np.isfinite(difference_standard_error) and difference_standard_error > 0:
        z_value = difference / difference_standard_error
        p_value = math.erfc(abs(z_value) / math.sqrt(2.0))
    else:
        z_value = np.nan
        p_value = np.nan
    stars = significance_stars(p_value)

    if not np.isfinite(p_value):
        inference = "not_available"
    elif p_value < 0.10 and difference > 0:
        inference = "state_2_significantly_higher"
    elif p_value < 0.10 and difference < 0:
        inference = "state_1_significantly_higher"
    else:
        inference = "no_statistically_significant_difference"

    return {
        "state_1_estimate": state_1_estimate,
        "state_1_standard_error": state_1_standard_error,
        "state_2_estimate": state_2_estimate,
        "state_2_standard_error": state_2_standard_error,
        "difference_state2_minus_state1": difference,
        "difference_standard_error": difference_standard_error,
        "z_value": z_value,
        "p_value": p_value,
        "significance_stars": stars,
        "difference_with_stars": f"{difference:.6f}{stars}" if np.isfinite(difference) else "",
        "inference": inference,
        "n_household_clusters": n_clusters,
    }

def _collapse_profile_statistics_to_households(
    annual: pd.DataFrame,
    gamma_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Align the estimation sample and collapse posterior-weighted statistics by household."""
    gamma = gamma_df[[
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2"
    ]].copy()
    if gamma.duplicated(["household_code", "panel_year"]).any():
        raise ValueError("Posterior gamma contains duplicate household-year keys.")

    aligned = annual.merge(
        gamma,
        on=["household_code", "panel_year"],
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    alignment_summary = pd.DataFrame([{
        "panel_household_years_before_alignment": int(len(annual)),
        "panel_household_years_missing_from_posterior": int((aligned["_merge"] == "left_only").sum()),
        "posterior_household_years_missing_from_panel": int((aligned["_merge"] == "right_only").sum()),
        "matched_household_years_retained": int((aligned["_merge"] == "both").sum()),
    }])
    matched = aligned.loc[aligned["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
    if matched.empty:
        raise ValueError("No household-years remain after profile/posterior alignment.")

    excluded = {
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2",
        *TRANSITION_EVENT_COLUMNS,
    }
    stat_columns = [column for column in annual.columns if column not in excluded]
    gamma_state_1 = matched["gamma_state_1"].to_numpy(dtype=np.float64, copy=False)
    gamma_state_2 = matched["gamma_state_2"].to_numpy(dtype=np.float64, copy=False)
    weighted = {"household_code": matched["household_code"]}
    weighted_columns: List[str] = []
    for column in stat_columns:
        values = matched[column].to_numpy(dtype=np.float64, copy=False)
        weighted[f"{column}_s1"] = values * gamma_state_1
        weighted[f"{column}_s2"] = values * gamma_state_2
        weighted_columns.extend([f"{column}_s1", f"{column}_s2"])
    household_stats = (
        pd.DataFrame(weighted)
        .groupby("household_code", sort=False, observed=True)[weighted_columns]
        .sum()
    )
    alignment_summary["matched_households_retained"] = int(len(household_stats))
    return household_stats, alignment_summary

def share_profile_outputs(result):
    """Both ratio estimands, clustered by household with fixed posterior weights."""
    hh,alignment=_collapse_profile_statistics_to_households(result.prepared.annual_stats,result.gamma_df)
    rows=[]
    for c in CATEGORY_NAMES:
        for metric,num,den in [
            ('mean_trip_healthy_food_share',c+'_share_sum','food_positive_n'),
            ('pooled_healthy_food_spending_share',c+'_healthy_spend_sum','food_total_spend_sum')]:
            row=dict(category=c,metric=metric,**infer_posterior_ratio_difference(hh,num,den))
            d=row['difference_state2_minus_state1']
            row['signed_difference_with_stars']=f'{d:+.6f}{row["significance_stars"]}' if np.isfinite(d) else ''
            se=row['difference_standard_error']
            row['difference_ci95_lower']=d-norm.ppf(.975)*se
            row['difference_ci95_upper']=d+norm.ppf(.975)*se
            row['difference_percentage_points']=100*d
            row['difference_se_percentage_points']=100*se
            row['difference_pp_with_stars']=f'{100*d:+.3f}{row["significance_stars"]}' if np.isfinite(d) else ''
            for state in (1,2):
                estimate=row[f'state_{state}_estimate']; stderr=row[f'state_{state}_standard_error']
                row[f'state_{state}_percent_with_se']=f'{100*estimate:.3f} ({100*stderr:.3f})'
            row['null_hypothesis']='State 2 mean share - State 1 mean share = 0'
            row['test']='two-sided normal Wald; unadjusted across categories'
            row['inference_status']='approximate_fixed_posteriors' if result.converged else 'provisional_nonconvergence'
            row['inference_method']='household-cluster delta method; posterior weights fixed'
            rows.append(row)
    return pd.DataFrame(rows),alignment


def model_implied_share_profiles(result):
    """Counterfactual state means using the SAME positive-food-spending trip X weights."""
    arrays=result.prepared.arrays; U=arrays._share_design; mapping=arrays._share_design_pos
    rows=[]
    for c in CATEGORY_NAMES:
        weights=np.bincount(mapping,weights=result.prepared.hy['food_positive_n'],minlength=len(U))
        for s in (1,2):
            X=np.column_stack([U[:,0],np.full(len(U),s),U[:,1:]])
            q=expit(X@result.gate_params[c+'_interior']['coef'])
            r=expit(X@result.gate_params[c+'_boundary_one']['coef']) if c+'_boundary_one' in result.gate_params else np.zeros(len(X))
            mu,_,_,phi,_,_=beta_terms(result.gate_params[c+'_beta']['coef'],X)
            rows.append(dict(category=c,state=s,
                expected_healthy_share=float(np.average((1-q)*r+q*mu,weights=weights)),
                probability_zero=float(np.average((1-q)*(1-r),weights=weights)),
                probability_one=float(np.average((1-q)*r,weights=weights)),
                probability_interior=float(np.average(q,weights=weights)),
                interior_mean_standardized=float(np.average(mu,weights=weights)),precision=float(phi),
                standardization='common empirical positive-food-spending trip covariate distribution',
                inference='point estimates only; whole-household refit bootstrap required for full uncertainty'))
    return pd.DataFrame(rows)


In [ ]:
def fitted_blocks(result, proposed):
    blocks=[]
    if proposed:
        blocks += [(f'transition_{name}',fit,True) for name,fit in result.transition_fits.items()]
    blocks += [(f'share_{k}',fit,True) for k,fit in result.gate_params.items()]
    return blocks


def coefficient_outputs(result, proposed):
    """Every free parameter exactly once; suppress unidentified Wald inference."""
    frames=[]; covariances=[]
    for name,fit,latent in fitted_blocks(result,proposed):
        t=fit['table'].copy().rename(columns={'std_error':'standard_error','z_value':'z_statistic'})
        t.insert(0,'equation_name',name)
        rank=fit['diagnostics']['hessian_rank']; k=len(fit['coef'])
        valid=rank==k and fit['diagnostics']['covariance_hessian_success']
        cov=fit['cov'].copy()
        if not valid:
            t['standard_error']=np.nan; cov[:]=np.nan
        t.loc[t.standard_error<=0,'standard_error']=np.nan
        t['hessian_based_variance']=t.standard_error**2
        t['z_statistic']=t.estimate/t.standard_error
        t['p_value']=2*norm.sf(abs(t.z_statistic))
        t['significance_stars']=t.p_value.map(significance_stars)
        t['ci_95_lower']=t.estimate-norm.ppf(.975)*t.standard_error
        t['ci_95_upper']=t.estimate+norm.ppf(.975)*t.standard_error
        t['estimate_with_stars']=[f'{v:.6f}{s}' for v,s in zip(t.estimate,t.significance_stars)]
        t['hessian_rank']=rank; t['block_parameter_count']=k
        t['covariance_method']='block M-step Hessian; posterior weights fixed' if latent else 'separable incidence Hessian'
        t['inference_status']='unavailable_rank_or_covariance' if not valid else ('provisional_nonconvergence' if not result.converged else 'approximate' if latent else 'model_based')
        frames.append(t); covariances.append(cov)
    table=pd.concat(frames,ignore_index=True)
    m=sum(k.endswith('_boundary_one') for k in result.gate_params)
    expected=78+6*m+(14 if proposed else 0)
    assert len(table)==expected and not table.parameter.duplicated().any()
    cov=np.full((expected,expected),np.nan); offset=0
    for block in covariances:
        k=len(block); cov[offset:offset+k,offset:offset+k]=block; offset+=k
    return table,pd.DataFrame(cov,index=table.parameter,columns=table.parameter)


def save_outputs(proposed,nested,destination):
    """Organized machine-readable and reviewable outputs for both fitted models."""
    root=Path(destination); root.mkdir(parents=True,exist_ok=True)
    comparison=[]
    for name,result,is_proposed in [('proposed_hmm',proposed,True),('nested_no_switch',nested,False)]:
        folder=root/('02_'+name if is_proposed else '03_'+name); folder.mkdir(exist_ok=True)
        coef,cov=coefficient_outputs(result,is_proposed)
        coef.to_csv(folder/'01_full_coefficient_table.csv',index=False)
        cov.to_csv(folder/'02_block_hessian_covariance.csv',index_label='parameter')
        result.gamma_df.rename(columns={'household_code':'hhid'}).to_csv(folder/'06_posterior_household_year.csv',index=False)
        result.history_df.to_csv(folder/'07_estimation_log.csv',index=False)
        diagnostics=pd.DataFrame([f['diagnostics'] for _,f,_ in fitted_blocks(result,is_proposed)])
        diagnostics.to_csv(folder/'09_optimizer_diagnostics.csv',index=False)
        n=len(result.prepared.arrays.hh_starts); ll=result.log_likelihood; k=len(coef)
        row=dict(model=name,log_likelihood=ll,n_parameters=k,AIC=-2*ll+2*k,BIC=-2*ll+k*np.log(n),
            n_households=n,n_household_years=len(result.prepared.hy),n_trips=len(result.prepared.df),
            fixed_initial_pi=result.pi_state2,converged=result.converged,stop_reason=result.stop_reason,
            em_iterations=int(result.history_df.iteration.max()),estimation_seconds=result.estimation_seconds,
            BIC_sample_size_definition='independent households',
            rank_deficient_blocks=int((diagnostics.hessian_rank<diagnostics.parameter_count).sum()))
        comparison.append(row)
        pd.DataFrame([row]).to_csv(folder/'05_model_fit.csv',index=False)
        profiles,alignment=share_profile_outputs(result)
        for metric in profiles.metric.unique():
            profiles.loc[profiles.metric==metric].to_csv(folder/('04_'+metric+'.csv'),index=False)
        primary=profiles.loc[profiles.metric=='mean_trip_healthy_food_share'].copy()
        primary.to_csv(folder/'04_posterior_healthy_alternative_share_significance.csv',index=False)
        readable=primary[['category','state_1_percent_with_se','state_2_percent_with_se',
            'difference_pp_with_stars','p_value','significance_stars']].rename(columns={
            'state_1_percent_with_se':'State 1 percent (SE)',
            'state_2_percent_with_se':'State 2 percent (SE)',
            'difference_pp_with_stars':'State 2 - State 1 (pp)'})
        readable.to_csv(folder/'04_posterior_healthy_share_readable.csv',index=False)
        print('\n'+name+': posterior healthy-alternative / food-spending shares')
        print(readable.to_string(index=False))
        print('Stars test State 2 - State 1; * p<.10, ** p<.05, *** p<.01. SEs hold posteriors fixed.')
        alignment.to_csv(folder/'profile_alignment.csv',index=False)
        implied=model_implied_share_profiles(result)
        implied.to_csv(folder/'08_model_implied_healthy_shares.csv',index=False)
        diagnostic=profiles.loc[profiles.metric=='mean_trip_healthy_food_share',
            ['category','state_1_estimate','state_2_estimate','difference_state2_minus_state1']].copy()
        diagnostic['state_2_higher_healthy_share']=diagnostic.difference_state2_minus_state1>0
        diagnostic.to_csv(folder/'10_state_label_diagnostic.csv',index=False)
        if not diagnostic.state_2_higher_healthy_share.all():
            warnings.warn(f'{name}: State 2 is not higher in all six categories. Review profiles before labeling Healthy.')
        if is_proposed:
            baseline,events=build_transition_outputs(result)
            baseline['event']='baseline_at_mean_events'
            pd.concat([baseline,events],ignore_index=True).to_csv(folder/'03_transition_matrices_by_event.csv',index=False)
        else:
            pd.DataFrame(np.eye(2),index=['from_state_1','from_state_2'],columns=['to_state_1','to_state_2']).to_csv(folder/'03_fixed_transition_matrix.csv')
    comp=pd.DataFrame(comparison)
    for m in ['AIC','BIC']: comp['delta_'+m]=comp[m]-comp[m].min()
    provisional=(not comp.converged.all()) or proposed.log_likelihood<nested.log_likelihood-TOL or comp.rank_deficient_blocks.gt(0).any()
    comp['comparison_status']='provisional: inspect convergence, likelihood and Hessians' if provisional else 'EM criteria met; local optima remain possible'
    folder=root/'01_model_comparison'; folder.mkdir(exist_ok=True)
    comp.to_csv(folder/'nested_vs_proposed.csv',index=False)
    pd.DataFrame([proposed.prepared.df.attrs['sorting_audit']]).to_csv(root/'sample_audit.csv',index=False)
    config=dict(version='V36',spending_absolute_tolerance=SPENDING_ATOL,denominator='food_spend',emission='zero_and_one_inflated_beta_with_absent_ones_fixed_zero',
        boundary_one_support=proposed.prepared.arrays._share_audit.to_dict(orient='records'),category_specification=CATEGORY_CONFIG,
        mean_epsilon=BETA_MEAN_EPS,log_precision_bounds=LOG_PRECISION_BOUNDS,input_path=str(INPUT_PATH),run_mode=RUN_MODE,transition_covariates=TRANSITION_EVENT_COLUMNS,
        share_covariates=GATE_COVARIATE_COLS,fixed_pi=proposed.pi_state2,minimum_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        gap_policy=YEAR_GAP_POLICY,max_em_iters=MAX_EM_ITERS,absolute_tolerance=TOL,relative_tolerance=EM_REL_TOL,
        numpy_version=np.__version__,pandas_version=pd.__version__,duckdb_version=duckdb.__version__)
    (root/'run_configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    (root/'README.md').write_text("""# V36 healthy-alternative / food-spending shares
All six outcomes use food_spend, on every positive-food-spending trip. No category-incidence equations.
No healthy purchase = zero share, even if the category is not bought. Zero-food trips have neutral emissions.
The same annual states, events, fixed initial probability and no-switch comparison are retained.
Nested free parameters = 78 + 6*m; proposed = 92 + 6*m, where m is the number of categories with 100% shares.
Absent 100% components are fixed at zero in both models; support decisions are in share_data_audit.csv.
04_posterior_healthy_alternative_share_significance.csv is the main mean-trip-share significance table.
04_posterior_healthy_share_readable.csv shows percentages (SEs) and percentage-point differences with stars.
Both models also export pooled healthy / food spending ratios as a separate estimand.
Stars test State 2 minus State 1, two-sided: * p<.10, ** p<.05, *** p<.01. P-values are unadjusted.
Difference SEs include cross-state covariance within households; posterior weights are treated as fixed.
Coefficient SEs are block M-step Hessians, not full observed-data HMM SEs. Cross-block covariance is unavailable.
For full uncertainty, refit complete household sequences in a bootstrap. Model-implied share profiles are point estimates.
The product emission assumes local independence and does not enforce the joint food-budget constraint.
LL/AIC/BIC compare models only within V36; they are not directly comparable with V35/V30 outcomes.
Inspect convergence, Hessian and state-label diagnostics. Full data were not run during development.
""",encoding='utf-8')
    print(comp[['model','log_likelihood','n_parameters','AIC','BIC','converged']].to_string(index=False))
    return comp


## 5. Execute
Set `INPUT_PATH`, then run all cells. `RUN_MODE='validate_only'` exports spending/support audits; `'synthetic'` runs an artificial fixture; `'full_estimation'` fits your data. Results go to `V36_Healthy_Food_Share_HMM_Results`. Both models print and export the requested share-difference significance table.


In [ ]:
def make_synthetic_parquet(path, n_households=120, seed=SEED, include_ones=True):
    """Artificial food budgets; includes zero-food trips and optional FV-only 100% trips."""
    rng=np.random.default_rng(seed); rows=[]; trip=0
    for h in range(n_households):
        state=int(rng.random()<.3)
        for t in range(4):
            events=rng.binomial(1,.15,6)
            if t: state=int(rng.random()<(.8 if state else .2))
            controls=[rng.binomial(1,.5),rng.uniform(1,12),rng.integers(1,6),rng.uniform(25,75)]
            for j in range(12):
                trip+=1; food=float(rng.integers(30,160))
                row=dict(hhid=f'{h:06d}',panel_year=2008+t,trip_id=str(trip),
                    purchase_date=pd.Timestamp(2008+t,1,1)+pd.Timedelta(days=j*20),
                    food_spend=food,initial_fresh_fv_share_2007=.3)
                row.update(zip(TRANSITION_EVENT_COLUMNS,events)); row.update(zip(GATE_COVARIATE_COLS,controls))
                active=rng.random(6)<(.8 if state else .3)
                amounts=rng.gamma(2.,1.,6)*active
                shares=amounts/(amounts.sum()+(8. if state else 20.))
                if include_ones and rng.random()<.03:
                    shares[:]=0.; shares[0]=1.
                if rng.random()<.02:
                    row['food_spend']=0.; food=0.
                for c,share in zip(CATEGORY_NAMES,shares):
                    row[CATEGORY_CONFIG[c]['healthy']]=food*share
                rows.append(row)
    pd.DataFrame(rows).to_parquet(path,index=False)


started=time.perf_counter()
if RUN_MODE not in ('full_estimation','validate_only','synthetic'): raise ValueError('Unknown RUN_MODE')
if RUN_MODE=='synthetic':
    destination=Path.cwd()/'V36_Synthetic_Results'; destination.mkdir(exist_ok=True)
    source=destination/'synthetic_input.parquet'
    make_synthetic_parquet(source)
    iterations=SYNTHETIC_EM_ITERS
else:
    source=INPUT_PATH; destination=OUTPUT_DIR; iterations=MAX_EM_ITERS
    if not source.is_file(): raise FileNotFoundError(source)
prepared=prepare_parquet(source,destination/'duckdb_temp')
print(f'Preparation: {time.perf_counter()-started:.1f} seconds')
if RUN_MODE=='validate_only':
    prepared.arrays._connection.close()
    destination.mkdir(parents=True,exist_ok=True)
    pd.DataFrame([prepared.df.attrs['sorting_audit']]).to_csv(destination/'sample_audit.csv',index=False)
else:
    proposed=estimate_model(prepared,max_em_iters=iterations)
    nested=estimate_no_switch_model(proposed,max_em_iters=iterations)
    comparison=save_outputs(proposed,nested,destination)
    assert proposed.pi_state2==nested.pi_state2==float(prepared.hy[FRESH_SHARE_2007_COL].iloc[0])
    print(f'Total elapsed: {time.perf_counter()-started:.1f} seconds. Results: {destination.resolve()}')
